# KALIA X19 — is it the gate, or the branch structure?

X18's `micro-gated` arm beat control by **0.0436 nats**, 4.4× the registered
bar — the largest effect any architecture arm has produced here. It was also
unattributable, because the gate was measured as non-responsive to its input
(G-0: std across inputs 5e-06 against a mean of 0.019).

So this trains **one arm**: the same four-branch normalisation with the gate
removed entirely.

| arm | params | val loss | source |
|---|---|---|---|
| control (`micro-base`) | 29,920,512 | 4.7662 | X18, not re-run |
| gated (`micro-gated`) | 30,072,768 | 4.7226 | X18, not re-run |
| **branchnorm** (`micro-branchnorm`) | 29,922,816 | *this run* | X19 |

Control and gated are **not re-run**: same seed, data, steps and lineage, and
their logs are published. Re-running would cost quota to reproduce numbers we
have, and would replace a fixed reference with a fresh draw.

**G-1 bar: val loss ≤ 4.7544** (midpoint of control and gated, +0.010).

If branchnorm clears that, the −0.0436 is the branch structure and the gate's
149,952 parameters are being spent on a constant. If it does not, the gated
arm's result was an artifact and the whole idea goes.

In [ ]:
import glob, os, shutil, subprocess, sys

work = '/kaggle/working/kalia'
if not os.path.exists(work):
    hits = sorted(glob.glob("/kaggle/input/**/ablate.py", recursive=True))
    assert hits, "attach the kalia-code-dev dataset (must be current: it carries configs/ and eval/)"
    shutil.copytree(os.path.dirname(hits[0]), work)
os.chdir(work)
print('code from', work)
print('branch_norm in model.py:', 'class BranchNorm' in open('model.py').read())
data = sorted(glob.glob('/kaggle/input/**/train.bin', recursive=True))
assert data, 'train.bin not found - attach the kalia-prep-v2b output'
print('data:', data[0])
cfg = open('configs/micro-branchnorm.yaml').read() if os.path.exists('configs/micro-branchnorm.yaml') else ''
print('arm config present:', bool(cfg))

In [ ]:
print(subprocess.run([sys.executable, 'ablate.py',
                       '--arms', 'micro-branchnorm',
                       '--steps', '500',
                       '--data-dir', str(os.path.dirname(data[0])),
                       '--out-root', '/kaggle/working/out'],
                      capture_output=True, text=True).stdout[-4000:])

In [ ]:
import pandas as pd
p = '/kaggle/working/out/micro-branchnorm/val_log.csv'
df = pd.read_csv(p)
print('micro-branchnorm: ' + '  '.join(f"{int(r.step)}={r.val_loss:.4f}" for r in df.itertuples()))
loss = float(df.iloc[-1].val_loss)
print(f'\nfinal val loss {loss:.4f}')

In [ ]:
# G-2 re-asserted on the checkpoint that actually trained, not on a
# freshly constructed model. Amendment 1 corrected the expected count from
# 29,920,512 to 29,922,816 (RMSNorm is affine, so the branch norm is not
# free) and the gate's own share to 149,952.
import torch
ck = torch.load('/kaggle/working/out/micro-branchnorm/ckpt.pt',
                 map_location='cpu', weights_only=False)
sd = ck['model']
# Count *parameters*, not state-dict entries. lm_head is tied to tok_emb, so
# summing sd.values() counts the embedding twice: 29,922,816 + 19,298,688 =
# 49,221,504, which is what version 1 of this kernel reported and called a
# FAIL. The model was correct; the check was not.
n_params = sum(v.numel() for k, v in sd.items() if k != 'lm_head.weight')
EXPECTED = 29922816
print(f'G-2: checkpoint params {n_params:,} | expected {EXPECTED:,} | '
      + ('PASS' if n_params == EXPECTED else 'FAIL'))
gates = [k for k in sd if '.gated.' in k]
branches = [k for k in sd if '.branch.' in k]
print(f'G-2: gate tensors present: {len(gates)} (expected 0 for this arm)')
print(f'G-2: branch tensors present: {len(branches)}')
assert not gates, 'this arm must carry no gate parameters'
assert n_params == EXPECTED, f'parameter count {n_params} != {EXPECTED}'
print('G-2 PASS')

In [ ]:
CONTROL, GATED, BAR = 4.7662, 4.7226, 4.7544
print('=== X19 G-1 ===')
print(f'  control      {CONTROL:.4f}')
print(f'  gated        {GATED:.4f}   ({GATED-CONTROL:+.4f} vs control)')
print(f'  branchnorm   {loss:.4f}   ({loss-CONTROL:+.4f} vs control, {loss-GATED:+.4f} vs gated)')
print(f'  G-1 bar      {BAR:.4f}   (midpoint + 0.010)')
print()
if loss <= BAR:
    print(f'  G-1 PASS  branchnorm captures >= half the gated arm gain')
    print('  => the -0.0436 is the BRANCH STRUCTURE, not the gate.')
    print('  => the gate spends 149,952 parameters on a constant (G-0).')
    print('  => promote branch normalisation to v0.3.0; drop the gate.')
else:
    print(f'  G-1 FAIL  branchnorm is {loss-BAR:+.4f} outside the bar')
    print('  => the gain is NOT explained by the branch structure alone,')
    print('     so either the gate does something G-0 could not detect, or')
    print('     X18s -0.0436 was a single-seed artifact. Both are negative')
    print('     results and are reported as such.')
print()
print('Caveat carried from X18 and still binding here: single seed, 30M '
      'parameters, 500 steps. A screening window, not a training run.')

Read alongside X18, not on its own.

- **G-1 pass** → X18's gain is attributed: branch normalisation, gate inert.
  A free architectural change and a 149,952-parameter deletion.
- **G-1 fail** → the branch structure does not explain the gain. Combined with
  G-0's zero input-dependence, that leaves X18's −0.0436 most likely a
  single-seed artifact, which would be the honest conclusion and would mean
  the gate *and* the branch norm both go.

Either way the registered thresholds are reported as written. G-1 is a
loss threshold in nats, so unlike X18's accuracy bar it is not sitting below
the measurement's own noise (see D45).